# Transform Circuits Data

1. Read bronze `circuits` table
2. Keep only the columns required for analytics (Drop `url` column)
3. Standardise column names using snake_case
4. Rename Columns to make them more meaningful
5. Filter out rows where `circuit_id` is null (business key validation)
6. Remove duplicate records
7. Transform values of columns `circuit_name` and `locality` to Title Case
8. Write the transformed data to silver `circuits` table


In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table = f'{catalog_name}.{bronze_schema}.circuits'
silver_table = f'{catalog_name}.{silver_schema}.circuits'

### Step 1 - Read bronze `circuits` table

In [0]:
# Using version
# circuits_df = spark.read.option('versionAsOf', 0).table(bronze_table) 

In [0]:
circuits_df = spark.table(bronze_table)

In [0]:
display(circuits_df)

### Step 2 - Keep only the columns required for analytics (Drop url column)

In [0]:
# Select columns 
# circuits_selected_df = circuits_df.select(
#     "circuitID",
#     "circuitName",
#     "lat",
#     "long",
#     "locality",
#     "country",
#     "ingestion_timestamp",
#     "source_file"
#     )

In [0]:
# Or
from pyspark.sql import functions as F

In [0]:
# by column experessions method
# we have some flexibility during selecting like alias name
circuits_selected_df = circuits_df.select(
    F.col("circuitID"),
    F.col("circuitName"),
    F.col("lat"),
    F.col("long"),
    F.col("locality"),
    F.col("country").alias('country_name'),     # Give alias name -> .alias('country_name')
    F.col("ingestion_timestamp"),
    F.col("source_file")
    )

In [0]:
display(circuits_selected_df)

### Step 3 & 4 - Standardise column names

- Standardize column names using snake_case
- Rename columns to make them more meaningful

In [0]:
circuits_renamed_df = (
    circuits_selected_df
        .withColumnRenamed('circuitID', 'circiut_id')
        .withColumnRenamed('circuitName', 'circuit_name')
        .withColumnRenamed('lat', 'latitude')
        .withColumnRenamed('long', 'longitude')
    )

OR

In [0]:
circuits_renamed_df = (
    circuits_selected_df
        .withColumnsRenamed({
            'circuitID': 'circuit_id',
            'circuitName': 'circuit_name',
            'lat': 'latitude',
            'long': 'longitude'
        })
    )

In [0]:
display(circuits_renamed_df)

### Step 5 - Filter out rows where circuit_id is null (business key validation)

In [0]:
# circuits_valid_df = circuits_renamed_df.filter(
#     "circuit_id IS NOT NULL"
# )

In [0]:
circuits_valid_df = circuits_renamed_df.filter(
    F.col('circuit_id').isNotNull()
)

In [0]:
display(circuits_valid_df)

### Step 6 - Remove duplicate records

In [0]:
# circuits_distinct_df = circuits_valid_df.distinct()

OR

In [0]:
circuits_distinct_df = circuits_valid_df.dropDuplicates(['circuit_id']) # specify column name without duplicate

In [0]:
display(circuits_distinct_df)

### Step 7 - Transform values of columns `circuit_name` and `locality` to Title Case

In [0]:
circuits_final_df = (
    circuits_distinct_df
        .withColumn('circuit_name', F.initcap(F.col("circuit_name")))
        .withColumn('locality', F.initcap(F.col("locality")))
)

In [0]:
display(circuits_final_df)

### Step 8 - Write the transformed data to silver `circuits` table

In [0]:
(
    circuits_final_df
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))